# 05_guest_gated_by_event — GuestRole gates the question, access_decide gates the answer

`@check_roles(GuestRole)` only means "anyone may ask this question" — it says nothing about what the answer is. `TrackOrderAction` below lets a guest track a package by a `tracking_token` from an email, with no login at all — but `access_decide` (level 3) still bases `allowed` on real facts: the token must match the specific order, AND the order must have actually reached the `"shipped"` event. Before that event happens, even a guest holding the exact right token gets an honest `allowed: false` — not because they are the wrong person, but because the event has not happened yet.

`access_decide` is not limited to "is this your object" checks — it can condition `allowed` on anything computable from `params`/`context`/`connections`: here, time/state, not ownership.

**What's new**

| Concept | Description |
|---|---|
| `GuestRole` gates the question | Anyone may ask; it does not decide the answer |
| `access_decide` on state/time | Conditions `allowed` on an event (`order.status`), not just ownership |

In [ ]:
!pip install aoa-action-machine aoa-fastapi-adapter

In [ ]:
from dataclasses import dataclass

from pydantic import Field

from aoa.action_machine.auth.guest_role import GuestRole
from aoa.action_machine.context.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.fastapi.permissions import to_wire

## A tiny in-memory "orders_db" stand-in

In [ ]:
class StoreDomain(BaseDomain):
    name = "store"
    description = "Store domain"


@dataclass(frozen=True)
class Order:
    tracking_token: str
    status: str  # "created" | "paid" | "shipped" | "in_transit" | "delivered"


orders = {7: Order(tracking_token="secret-token-7", status="paid")}

## TrackOrderAction — GuestRole, but access_decide checks token + event

In [ ]:
class TrackOrderParams(BaseParams):
    order_id: int = Field(description="Order identifier")
    tracking_token: str = Field(description="Secret token from the shipping confirmation email")


class TrackOrderResult(BaseResult):
    status: str = Field(description="Order status")


@meta(description="Track an order by its tracking token", domain=StoreDomain)
@check_roles(GuestRole)  # role-gate: anyone may ask — access_decide still decides the answer
class TrackOrderAction(BaseAction[TrackOrderParams, TrackOrderResult]):

    async def access_decide(self, params, context, box, connections) -> bool:
        order = orders[params.order_id]
        if order.tracking_token != params.tracking_token:
            return False  # wrong secret — not this order at all
        return order.status in ("shipped", "in_transit", "delivered")

    @summary_aspect("Return the order status")
    async def track_summary(self, params, state, box, connections):
        return TrackOrderResult(status=orders[params.order_id].status)

## Before and after the "shipped" event

Same guest, same token, no client-side change — only the real-world event flips the answer.

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
machine = ActionProductMachine()
guest = Context()  # no login at all — GuestRole covers this
params = TrackOrderParams(order_id=7, tracking_token="secret-token-7")

before = to_wire(await machine.check_access_decide(guest, TrackOrderAction, params))
print(f"before shipped: allowed={before.allowed}")

orders[7] = Order(tracking_token="secret-token-7", status="shipped")  # the real-world event fires

after = to_wire(await machine.check_access_decide(guest, TrackOrderAction, params))
print(f"after shipped:  allowed={after.allowed}")